# Nutrition Assistant development demo

This notebook exercises the canonical package APIs in a top-to-bottom development workflow. Install the project in editable mode before running it (`python -m pip install -e .`).


## Imports

Application behavior lives in `src/nutrition_assistant`; this notebook only composes those APIs.


In [ ]:
from nutrition_assistant.config import DATABASE_PATH
from nutrition_assistant.database import initialize_database
from nutrition_assistant.engine.nutrition_engine import NutritionEngine
from nutrition_assistant.models.ingredient import Ingredient
from nutrition_assistant.models.meal import Meal
from nutrition_assistant.planner.optimizer import rank_meals
from nutrition_assistant.repositories.food_repository import FoodRepository
from nutrition_assistant.repositories.meal_repository import MealRepository
from nutrition_assistant.repositories.target_repository import TargetRepository


## Initialize and inspect the database

The canonical bootstrap loads the required USDA CSVs when needed and creates the application schema. Re-running it is safe.


In [ ]:
initialize_database()

food_repository = FoodRepository(DATABASE_PATH)
meal_repository = MealRepository(food_repository.con)
target_repository = TargetRepository(food_repository.con)
nutrition_engine = NutritionEngine(food_repository)

print(f"Database: {DATABASE_PATH}")
display(food_repository.con.execute("SHOW TABLES").fetchdf())


## Search for foods

Search for two ingredients and select a food from each result set for the examples below. Adjust the search terms or selected rows interactively when exploring other foods.


In [ ]:
oat_results = food_repository.search("oatmeal", limit=10)
banana_results = food_repository.search("banana", limit=10)

if oat_results.empty or banana_results.empty:
    raise RuntimeError("The demo searches did not return both required foods")

display(oat_results)
display(banana_results)

oat_fdc_id = int(oat_results.iloc[0]["fdc_id"])
banana_fdc_id = int(banana_results.iloc[0]["fdc_id"])


## Inspect portions and nutrients

USDA nutrient amounts are reported per 100 grams. Portion rows are shown separately so a specific portion can be selected when more than one is available.


In [ ]:
oat_portions = food_repository.get_portions(oat_fdc_id)
oat_nutrients = food_repository.get_nutrients(oat_fdc_id)
oat_nutrients_for_100g = nutrition_engine.calculate_food_nutrients(
    oat_fdc_id,
    grams=100,
)

display(oat_portions)
display(oat_nutrients.head(20))
display(oat_nutrients_for_100g.head(20))


## Create and load a meal

This saves one demonstration meal. Re-running this cell intentionally creates another saved prototype meal.


In [ ]:
demo_meal = Meal(
    name="Oatmeal and banana demo",
    ingredients=[
        Ingredient(fdc_id=oat_fdc_id, grams=80, name="Oatmeal"),
        Ingredient(fdc_id=banana_fdc_id, grams=120, name="Banana"),
    ],
)

demo_meal_id = meal_repository.create_meal(demo_meal)
loaded_meal = meal_repository.get_meal(demo_meal_id)

print(f"Saved meal_id: {demo_meal_id}")
display(loaded_meal)


## Calculate meal nutrients

Only nutrients reported by USDA are present; missing nutrient data is not converted to zero.


In [ ]:
meal_nutrients = nutrition_engine.calculate_meal(loaded_meal)
display(meal_nutrients)


## Load the default nutrient targets

The canonical default profile is seeded during database initialization and loaded through `TargetRepository`.


In [ ]:
default_targets = target_repository.get_profile("default")

display(default_targets)


## Calculate and summarize a day

For demonstration, the day contains the saved meal once and is scored against the seeded default targets.


In [ ]:
day_nutrients = nutrition_engine.calculate_day([loaded_meal])
day_score = nutrition_engine.score_against_targets(
    day_nutrients,
    default_targets,
)
day_summary = nutrition_engine.summarize_score(day_score)

display(day_nutrients)
display(day_summary)


## Rank candidate meals

The prototype planner ranks in-memory candidate meals against the current day score.


In [ ]:
candidate_meals = [
    Meal(
        name="Oatmeal candidate",
        ingredients=[Ingredient(oat_fdc_id, grams=100, name="Oatmeal")],
    ),
    Meal(
        name="Banana candidate",
        ingredients=[Ingredient(banana_fdc_id, grams=120, name="Banana")],
    ),
]

ranked_candidates = rank_meals(
    current_score=day_score,
    meals=candidate_meals,
    nutrition_engine=nutrition_engine,
)

for candidate in ranked_candidates:
    print(
        candidate["meal_name"],
        "score=",
        candidate["score"],
        "nutrients_helped=",
        candidate["nutrients_helped"],
    )


## Close the database connection


In [ ]:
food_repository.close()
